# 9.1 KV缓存 (KV Cache)

KV缓存是大语言模型推理优化的核心技术，通过缓存已计算的Key和Value避免重复计算。

本节涵盖：
- KV缓存原理
- MQA
- GQA
- MLA
- PagedAttention

## 1. KV缓存原理

**目的**：避免自回归生成时的重复计算

**基本原理**：在自回归生成中，每生成一个新token时，之前所有token的KV值不变。将已计算的KV缓存起来，新token只需计算自己的QKV，然后用Q与缓存的KV做注意力。

**显存分析**（7B模型，FP16）：
- 每token每层KV: 2 × d_model × 2 bytes
- 32层: 32 × 2 × 4096 × 2 = 512 KB/token
- 2048 token序列: ~1 GB KV缓存

In [ ]:
import torch
import math

d_model = 4096
n_layers = 32
bytes_per_param = 2
kv_per_token_per_layer = 2 * d_model * bytes_per_param
kv_per_token = n_layers * kv_per_token_per_layer

print('=== KV Cache Memory Analysis ===')
print(f'Model: 7B (d_model={d_model}, {n_layers} layers, FP16)')
print(f'KV per token per layer: {kv_per_token_per_layer/1024:.1f} KB')
print(f'KV per token (all layers): {kv_per_token/1024:.1f} KB')

for seq_len in [512, 1024, 2048, 4096, 8192]:
    total_kv = kv_per_token * seq_len
    print(f'  Seq len {seq_len:>5d}: {total_kv/1e9:.2f} GB KV cache')

batch_size = 32
seq_len = 2048
total = kv_per_token * seq_len * batch_size
print(f'\nBatch {batch_size}, Seq {seq_len}: {total/1e9:.1f} GB KV cache')
print(f'\nKey: KV cache grows linearly with sequence length and batch size.')
print(f'Long sequences + large batches = KV cache becomes the memory bottleneck.')

## 2. MQA / GQA / MLA

**MQA（Multi-Query Attention）**：所有注意力头共享同一组KV，KV头数为1，KV缓存减少为1/n_heads。

**GQA（Grouped-Query Attention）**：将注意力头分组，每组共享一组KV，是MQA和MHA的折中。

**MLA（Multi-head Latent Attention）**：通过低秩压缩将KV投影到低维潜在空间，大幅减少KV缓存。

**KV缓存对比**：
| 方法 | KV头数 | KV缓存比例 |
|------|--------|------------|
| MHA | n_heads | 100% |
| GQA | n_groups | n_groups/n_heads |
| MQA | 1 | 1/n_heads |
| MLA | 1 (compressed) | ~5% |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model=512, n_heads=8, n_kv_heads=2):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.d_head = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, -1)
        return self.out_proj(out)

mha_kv = 8 * 64 * 2
gqa_kv = 2 * 64 * 2
mqa_kv = 1 * 64 * 2

gqa = GroupedQueryAttention(512, 8, 2)
x = torch.randn(2, 16, 512)
out = gqa(x)

print('=== MQA / GQA / MLA ===')
print(f'Config: d_model=512, n_heads=8, d_head=64')
print(f'\nKV cache per token per layer (FP16):')
print(f'  MHA (8 KV heads): {mha_kv*2/1024:.1f} KB')
print(f'  GQA (2 KV heads): {gqa_kv*2/1024:.1f} KB ({gqa_kv/mha_kv:.0%} of MHA)')
print(f'  MQA (1 KV head):  {mqa_kv*2/1024:.1f} KB ({mqa_kv/mha_kv:.0%} of MHA)')
print(f'\nGQA output: {out.shape}')
print(f'\nKey: GQA is the sweet spot between quality and efficiency.')
print(f'Used by LLaMA-2/3, Mistral, Qwen, DeepSeek.')

## 3. PagedAttention

**目的**：高效管理KV缓存显存

**基本原理**：将KV缓存按固定大小的页（block）管理，类似操作系统的虚拟内存分页机制，消除显存碎片，支持动态显存分配。

**核心优势**：
- 消除显存碎片（传统预分配浪费60-80%显存）
- 支持不同长度的序列共享GPU显存
- 支持KV缓存的跨请求共享（如parallel sampling）

**代表**：vLLM

In [ ]:
import torch

block_size = 16
seq_lengths = [23, 47, 12, 88, 35]

print('=== PagedAttention ===')
print(f'Block size: {block_size} tokens')
print(f'\nSequence lengths and block allocation:')

total_blocks = 0
total_waste = 0
total_tokens = 0
for i, seq_len in enumerate(seq_lengths):
    n_blocks = (seq_len + block_size - 1) // block_size
    allocated = n_blocks * block_size
    waste = allocated - seq_len
    total_blocks += n_blocks
    total_waste += waste
    total_tokens += seq_len
    print(f'  Seq {i}: len={seq_len:>3d}, blocks={n_blocks}, allocated={allocated}, waste={waste}')

max_len = max(seq_lengths)
traditional_waste = sum(max_len - s for s in seq_lengths)

print(f'\nPagedAttention waste: {total_waste} tokens ({total_waste/(total_blocks*block_size):.1%})')
print(f'Traditional pre-alloc waste: {traditional_waste} tokens ({traditional_waste/(max_len*len(seq_lengths)):.1%})')
print(f'\nKey: PagedAttention reduces waste from {traditional_waste/(max_len*len(seq_lengths)):.0%} to {total_waste/(total_blocks*block_size):.0%}.')
print(f'vLLM achieves near-zero waste with paged KV cache management.')